In [2]:
import librosa
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
from tqdm import tqdm

print("Libraries loaded successfully")

Libraries loaded successfully


In [17]:
#output folder created for extracted features
PREPROCESSED_DIR = "../output/preprocessed/"
FEATURE_DIR = "../output/features/"
os.makedirs(FEATURE_DIR, exist_ok=True)

print("Feature output folder ready.")

Feature output folder ready.


In [18]:
"""
    Extract MFCC features from preprocessed audio signal.

    Parameters:
    ----------
    signal : numpy array
        Preprocessed audio waveform

    sr : int
        Sampling rate of audio

    n_mfcc : int
        Number of MFCC coefficients to keep
        Standard practice = 13

    Returns:
    -------
    mfcc_matrix : numpy array
        Shape = (num_frames, 13)
        Each row = one short audio frame
        Each column = one MFCC coefficient
    """
def extract_mfcc(signal, sr, n_mfcc=13):

    mfcc = librosa.feature.mfcc(
        y=signal,
        sr=sr,

        # Keep first 13 MFCC coefficients
        n_mfcc=n_mfcc,

        # FFT window size
        n_fft=1024,

        # Hop length = 512 gives 50% overlap
        hop_length=512,

        # Frame/window length
        win_length=1024,

        # Hamming window reduces spectral leakage
        window='hamming',

        # Number of Mel filters
        n_mels=40
    )

    # librosa returns shape (13, frames)
    # Transpose to (frames, 13) for easier interpretation
    return mfcc.T

In [19]:
  """
    Convert variable-length MFCC matrix into fixed-size vector.

    Why?
    ----
    Traditional ML models require same-size input for every sample.
    Since audio files have different durations, number of frames varies.

    Solution:
    --------
    Compute statistical summary across frames.

    Output:
    -------
    26-dimensional vector:
        13 means  +  13 variances
    """
def aggregate_mfcc(mfcc_matrix):

    # Mean of each MFCC coefficient across all frames
    mfcc_mean = np.mean(mfcc_matrix, axis=0)

    # Variance of each MFCC coefficient across all frames
    mfcc_var = np.var(mfcc_matrix, axis=0)

    # Concatenate into single feature vector
    feature_vector = np.concatenate([mfcc_mean, mfcc_var])

    return feature_vector

In [20]:
preprocessed_files = glob.glob(os.path.join(PREPROCESSED_DIR, "*.wav"))
print(f"Total files to process: {len(preprocessed_files)}")

all_features = []
file_names   = []


for i, filepath in enumerate(preprocessed_files):
    try:
        # Load preprocessed audio
        signal, sr = librosa.load(filepath, sr=None)

        # Extract MFCC matrix
        mfcc_matrix = extract_mfcc(signal, sr)

        # Convert to fixed-length feature vector
        feature_vector = aggregate_mfcc(mfcc_matrix)

        # Store
        all_features.append(feature_vector)
        file_names.append(os.path.basename(filepath))
    
    except Exception as e:
        # Add error handling here
        print(f"Error processing {filepath}: {e}")
    
    # Progress update every 50 files
    if (i + 1) % 50 == 0 or (i + 1) == len(preprocessed_files):
        print(f"[{i+1}/{len(preprocessed_files)}] MFCC extracted")

Total files to process: 920
[50/920] MFCC extracted
[100/920] MFCC extracted
[150/920] MFCC extracted
[200/920] MFCC extracted
[250/920] MFCC extracted
[300/920] MFCC extracted
[350/920] MFCC extracted
[400/920] MFCC extracted
[450/920] MFCC extracted
[500/920] MFCC extracted
[550/920] MFCC extracted
[600/920] MFCC extracted
[650/920] MFCC extracted
[700/920] MFCC extracted
[750/920] MFCC extracted
[800/920] MFCC extracted
[850/920] MFCC extracted
[900/920] MFCC extracted
[920/920] MFCC extracted


In [21]:
#feature list is converted to numpy array
X = np.array(all_features)

print("MFCC extraction complete.")
print("Final feature matrix shape:", X.shape)

MFCC extraction complete.
Final feature matrix shape: (920, 26)


In [22]:
#Save Extracted Features
np.save(os.path.join(FEATURE_DIR, "mfcc_features.npy"), X)
np.save(os.path.join(FEATURE_DIR, "file_names.npy"), np.array(file_names))

print("MFCC features saved successfully.")

MFCC features saved successfully.
